# Projeto Prático: Machine Learning & Inteligência de Mercado
## Análise Estratégica da Concentração no Comércio Global de Bens Criativos (Dataset OpenFCS)

---

> **Componente Curricular:** Machine Learning aplicado à Administração
> **Instituição:** Curso de Graduação em Administração
> **Objetivo:** Aplicação prática de Ciência de Dados, Machine Learning e Inteligência Artificial Generativa para diagnosticar padrões de concentração e (re)configuração competitiva no comércio mundial de bens criativos, a partir do acervo aberto OpenFCS (UNCTAD, alinhado ao UNESCO Framework for Cultural Statistics 2025).

---

### Corpo Docente & Contato

| Atributo | Detalhes |
| :--- | :--- |
| **Professor** | **Sérgio Assunção Monteiro, D.Sc.** |
| **Conecte-se no LinkedIn** | [🌐 linkedin.com/in/sergio-assunção-monteiro](https://www.linkedin.com/in/sergio-assun%C3%A7%C3%A3o-monteiro-b781897b/) |
| **Currículo Lattes** | [🔬 lattes.cnpq.br/9489191035734025](http://lattes.cnpq.br/9489191035734025) |
| **Repositório GitHub** | [💻 github.com/sergiomonteiro76](https://github.com/sergiomonteiro76) |

---

### Sobre este Notebook
Este ambiente foi configurado para que os alunos atuem como **Analistas de Inteligência de Mercado**. Ao longo do semestre, com apoio de modelos de linguagem (IA) integrados ao ecossistema do Google Colab, vamos reconstruir — do dado bruto ao modelo preditivo — o diagnóstico de estrutura competitiva de um setor econômico real: o comércio internacional de bens criativos.

* **Fonte de dados:** [OpenFCS Dataset](https://doi.org/10.5281/zenodo.21211053) — Monteiro & Dubeux (2026), CC-BY-4.0.
* **Material de apoio:** Capítulos 1 a 3 das notas de aula (Ambiente e primeiro contato; Python/pandas/NumPy; Bases de Dados e SQL).

## **Aula 4 — Qualidade e Preparação de Dados: Auditando o Funil de Limpeza**
O acervo OpenFCS não publica os 25,4 milhões de registros brutos da UNCTAD — apenas os dados já limpos e os artefatos de proveniência. Nesta aula, em vez de refazer o funil do zero, vamos auditá-lo: conferir se os números batem, inspecionar a lista de permissão por dentro, e aplicar técnicas de qualidade de dados (duplicatas, ausentes, outliers, tidy data) sobre a tabela já limpa.

### **Recarregar o acervo**

In [1]:
import requests, zipfile, io, os
import pandas as pd

url = "https://zenodo.org/records/21211053/files/openfcs_v1.0.0.zip?download=1"
resp = requests.get(url)
resp.raise_for_status()

with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    z.extractall("openfcs")

endereco = "openfcs/openfcs-1.0.0/data/derived/"
edges = pd.read_csv(endereco + "trade_edges.csv")
entities = pd.read_csv(endereco + "entities.csv")
crosswalk = pd.read_csv(endereco + "products_crosswalk.csv")

edges7 = edges[edges["resolution"] == "cer7"].copy()
print(f"edges7: {len(edges7):,} linhas")   # deve dar 1.026.400

!pip install duckdb --quiet
import duckdb

edges7: 1,026,400 linhas


### **4.1 Por que não refazemos o funil inteiro do zero**
A Tabela 1.1 do Capítulo 1 mostra a jornada de 25,4 milhões de registros brutos até 2.197.978 linhas limpas. Esse dado bruto não está no pacote publicado — só o resultado final e os artefatos que documentam como ele foi obtido. É assim que a maioria dos projetos de dados reais funciona: você raramente tem acesso ao dado bruto original de outra organização, mas pode (e deve) auditar o processo documentado.

In [2]:
for root, _, files in os.walk("openfcs"):
    for f in files:
        print(os.path.join(root, f))

openfcs/openfcs-1.0.0/.zenodo.json
openfcs/openfcs-1.0.0/CITATION.cff
openfcs/openfcs-1.0.0/requirements.txt
openfcs/openfcs-1.0.0/checksums.sha256
openfcs/openfcs-1.0.0/LICENSE
openfcs/openfcs-1.0.0/README.md
openfcs/openfcs-1.0.0/figures/Fig3.pdf
openfcs/openfcs-1.0.0/figures/figure_manifest.csv
openfcs/openfcs-1.0.0/figures/Fig2.tiff
openfcs/openfcs-1.0.0/figures/Fig1.pdf
openfcs/openfcs-1.0.0/figures/Fig3.tiff
openfcs/openfcs-1.0.0/figures/.gitkeep
openfcs/openfcs-1.0.0/figures/Fig2.pdf
openfcs/openfcs-1.0.0/figures/Fig1.tiff
openfcs/openfcs-1.0.0/diagnostics/profile_pdfs.py
openfcs/openfcs-1.0.0/diagnostics/investigate_findings.py
openfcs/openfcs-1.0.0/diagnostics/balanced_panel_test.py
openfcs/openfcs-1.0.0/diagnostics/profile_unctad.py
openfcs/openfcs-1.0.0/diagnostics/reports/balanced_panel_20260705T020259Z.json
openfcs/openfcs-1.0.0/diagnostics/reports/investigation_20260705T015259Z.json
openfcs/openfcs-1.0.0/diagnostics/reports/balanced_panel_20260705T020256Z.json
openfcs/ope

### **4.2 Auditando o manifesto de execução**

In [5]:
import json

# Ajuste o caminho abaixo conforme o que apareceu na Célula 4
caminho_manifest = endereco + "run_manifest.json"

with open(caminho_manifest) as f:
    manifest = json.load(f)

print(manifest.keys())

dict_keys(['tool', 'generated_utc', 'input_sha256', 'entities_sha256', 'config', 'funnel', 'n_economies_used', 'n_spectral_layers', 'self_check'])


### **self-check: o manifesto bate com a Tabela 1.1 do Capítulo 1?**

In [6]:
# Explore o conteudo do manifesto e procure as contagens do funil
# (nomes de chave a confirmar - inspecione "manifest" acima)
manifest

{'tool': 'openfcs_extract_v2.py',
 'generated_utc': '2026-07-05T01:43:02.384243+00:00',
 'input_sha256': '588173f1909ee938f55e133c33531916c745dc59dfcb07854f165ba5f72a0ca2',
 'entities_sha256': '2e1f4bc2b991f98939aa5a54f1dae6414b568953e8f6886f1c3257e6e264fc39',
 'config': {'resolutions': ['cer7', 'craft_sub'],
  'min_economies': 5,
  'years': None,
  'excluded_products': ['All creative goods']},
 'funnel': {'raw_rows': 25437609,
  'after_dropna_value': 18905124,
  'after_exports': 8993089,
  'after_product_map': 8057125,
  'after_allowlist_both_sides': 2198626,
  'after_drop_selfloops': 2197978,
  'final_clean_rows': 2197978},
 'n_economies_used': 242,
 'n_spectral_layers': 322,
 'self_check': {'edges_equals_final_clean': True,
  'layers_by_resolution': {'cer7': 161, 'craft_sub': 161},
  'n_layers_v1_used_negative_eig': 322}}

### **4.3 A lista de permissão por dentro**
`entities.csv` classifica **todos** os nomes encontrados no dado bruto — não só os 242 aprovados. Vamos inspecionar antes de assumir qualquer coluna.

In [7]:
print(entities.columns.tolist())
print(f"Total de linhas em entities: {len(entities)}")
entities.head(10)

['name', 'in_economy', 'in_partner', 'kind']
Total de linhas em entities: 309


,name,in_economy,in_partner,kind
0,Afghanistan,1,1,economy
1,Africa,1,1,aggregate
2,Albania,1,1,economy
3,Algeria,1,1,economy
4,American Samoa,0,1,economy
5,Americas,1,1,aggregate
6,Andorra,1,1,economy
7,Angola,1,1,economy
8,Anguilla,1,1,economy
9,Antigua and Barbuda,1,1,economy


### **individual vs. agregado — ajuste a coluna conforme a inspeção acima**

In [9]:
coluna_classificacao = "kind"
print(entities[coluna_classificacao].value_counts())

# A licao da Aula 2 do material de apoio: paises com nome de continente no meio
coluna_nome = "name"
suspeitos = entities[entities[coluna_nome].str.contains("Africa", case=False, na=False)]
suspeitos

kind
economy      242
aggregate     67
Name: count, dtype: int64


,name,in_economy,in_partner,kind
1,Africa,1,1,aggregate
48,Central African Republic,1,1,economy
83,Developing economies: Africa,1,1,aggregate
89,Developing economies: Northern Africa,1,1,aggregate
93,Developing economies: Sub-Saharan Africa,1,1,aggregate
98,Eastern Africa,1,1,aggregate
159,LDCs: Africa,1,1,aggregate
184,Middle Africa,1,1,aggregate
204,Northern Africa,1,1,aggregate
256,South Africa,1,1,economy


### **4.4 Duplicatas, valores ausentes e outliers na tabela já limpa**
Se o pipeline documentado funcionou, `edges7` não deveria ter duplicatas nem valores ausentes. Vamos confirmar — é uma forma de validar a limpeza, não de refazê-la.

In [10]:
print(f"Linhas duplicadas: {edges7.duplicated().sum()}")
print(f"\nValores ausentes por coluna:")
print(edges7.isna().sum())

Linhas duplicadas: 0

Valores ausentes por coluna:
edge_id               0
economy               0
partner               0
product               0
year                  0
value_usd_millions    0
flow                  0
resolution            0
cer_code              0
fcs_domain            0
mapping_status        0
dtype: int64


### **outliers com o método IQR**

In [11]:
Q1 = edges7["value_usd_millions"].quantile(0.25)
Q3 = edges7["value_usd_millions"].quantile(0.75)
IQR = Q3 - Q1
limite_superior = Q3 + 1.5 * IQR

outliers = edges7[edges7["value_usd_millions"] > limite_superior]
print(f"Limite superior (IQR): {limite_superior:,.2f}")
print(f"Fluxos acima do limite: {len(outliers):,} ({len(outliers)/len(edges7):.1%})")

outliers.sort_values("value_usd_millions", ascending=False).head(10)[
    ["economy", "partner", "fcs_domain", "year", "value_usd_millions"]
]

Limite superior (IQR): 1.26
Fluxos acima do limite: 189,555 (18.5%)


,economy,partner,fcs_domain,year,value_usd_millions
1998619,G-77 (Group of 77),United States,C. Visual arts (crafts) / F. Design,2022,82696.264
1896014,G-77 (Group of 77),United States,C. Visual arts (crafts) / F. Design,2021,81079.480
2197838,G-77 (Group of 77),United States,C. Visual arts (crafts) / F. Design,2024,71596.783
2101839,G-77 (Group of 77),United States,C. Visual arts (crafts) / F. Design,2023,69995.871
1913364,China,G-77 (Group of 77),C. Visual arts (crafts) / F. Design,2022,63834.370
2016097,China,G-77 (Group of 77),C. Visual arts (crafts) / F. Design,2023,61655.022
2118450,China,G-77 (Group of 77),C. Visual arts (crafts) / F. Design,2024,61274.620
1175547,G-77 (Group of 77),"China, Hong Kong SAR",C. Visual arts (crafts) / F. Design,2014,57871.537
1588926,G-77 (Group of 77),United States,C. Visual arts (crafts) / F. Design,2018,56689.367
1793397,G-77 (Group of 77),United States,C. Visual arts (crafts) / F. Design,2020,55534.422


### **4.5 Tidy vs. wide: a mesma tabela, duas formas**
`edges7` já está em formato *tidy* (uma linha = uma observação). Vamos transformá-la em *wide* — e depois voltar — para entender quando cada forma é útil.

In [12]:
# Wide: uma linha por economia, uma coluna por ano (NAO tidy)
wide = edges7.pivot_table(
    index="economy", columns="year", values="value_usd_millions", aggfunc="sum"
)
wide.head()

# De volta a tidy com melt
tidy_de_novo = wide.reset_index().melt(
    id_vars="economy", var_name="year", value_name="value_usd_millions"
)
tidy_de_novo.head()

,economy,year,value_usd_millions
0,Afghanistan,2002,NaN
1,Albania,2002,8.870
2,Algeria,2002,6.243
3,Andorra,2002,8.801
4,Angola,2002,NaN


### **4.6 Status de confirmação do crosswalk**

In [13]:
print(crosswalk.columns.tolist())
print(crosswalk["status"].value_counts())

['product', 'resolution', 'cer', 'fcs', 'status', 'parent']
status
provisional             8
confirmed               4
provisional_straddle    2
Name: count, dtype: int64


## 🧪 Exercícios Práticos — Aula 4

> **Como usar:** resolva cada exercício em uma célula de código abaixo do enunciado. Depois, leve o resultado para uma IA usando o *prompt sugerido*.

---

### Exercício 1 — Duplicatas e ausentes
**📝 Tarefa:** Confirme se `edges7` tem duplicatas ou valores ausentes. Se houver algum, investigue de onde vêm.

**🤖 Pergunte à IA:**
> "Confirmei que meu dataset tem [X] duplicatas e [Y] valores ausentes. O que isso sugere sobre a qualidade do pipeline de limpeza documentado?"

---

### Exercício 2 — Allowlist na prática
**📝 Tarefa:** Usando `entities.csv`, liste 5 exemplos de nomes classificados como "agregado" e explique por que cada um não é um país.

**🤖 Pergunte à IA:**
> "Estes são 5 nomes classificados como agregados em uma base de comércio internacional: [cole a lista]. Por que cada um não deveria ser tratado como um país individual em uma rede de comércio?"

---

### Exercício 3 — Outliers em dados econômicos
**📝 Tarefa:** Usando o método IQR, identifique os maiores outliers de `value_usd_millions`. Você removeria algum? Justifique.

**🤖 Pergunte à IA:**
> "Encontrei estes outliers em um dataset de comércio internacional: [cole os 5 maiores]. Em dados econômicos, um outlier estatístico é necessariamente um erro? Por quê?"

---

### Exercício 4 — Tidy vs. wide na prática de BI
**📝 Tarefa:** Transforme `edges7` em uma tabela *wide* (economia × ano) para um domínio específico. Em que situação de BI essa forma seria útil? E quando a forma *tidy* é obrigatória?

**🤖 Pergunte à IA:**
> "Tenho os mesmos dados em formato tidy e em formato wide (economia x ano). Para montar um dashboard no Power BI, qual formato devo usar como fonte, e por quê?"

---

### Exercício 5 — Auditoria do manifesto
**📝 Tarefa:** Compare os números do `run_manifest.json` com a Tabela 1.1 do Capítulo 1. Eles batem?

**🤖 Pergunte à IA:**
> "Comparei o funil documentado em um artigo científico com o manifesto de execução publicado junto ao dataset e [bateu / não bateu, explique]. Isso aumenta ou diminui minha confiança na reprodutibilidade do estudo?"